In [19]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
# 向量化表达与特征工程：人民公社知识图谱数据

import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
import re

# 读取数据（请替换为你导出的 df_all 数据路径）
df_all = pd.read_csv(r"E:\【论文】\heritage science\0609Chinese_knowledge_graph\commune_features.csv", encoding='utf-8-sig')

# 提取经纬度
def extract_lat_lon(location):
    match = re.search(r'\(?([0-9.]+),\s*([0-9.]+)\)?', str(location))
    if match:
        lat, lon = float(match.group(1)), float(match.group(2))
        return pd.Series({'lat': lat, 'lon': lon})
    return pd.Series({'lat': None, 'lon': None})

df_all[['lat', 'lon']] = df_all['location'].apply(extract_lat_lon)

# 替换空值为列表
for col in ['agri_attrs', 'ind_attrs', 'other_attrs']:
    df_all[col] = df_all[col].apply(lambda x: eval(x) if isinstance(x, str) else ([] if pd.isna(x) else x))

# One-hot 编码：农业
mlb_agri = MultiLabelBinarizer()
agri_df = pd.DataFrame(mlb_agri.fit_transform(df_all['agri_attrs']),
                       columns=[f"agri_{c}" for c in mlb_agri.classes_])

# 工业
mlb_ind = MultiLabelBinarizer()
ind_df = pd.DataFrame(mlb_ind.fit_transform(df_all['ind_attrs']),
                      columns=[f"ind_{c}" for c in mlb_ind.classes_])

# 其他
mlb_other = MultiLabelBinarizer()
other_df = pd.DataFrame(mlb_other.fit_transform(df_all['other_attrs']),
                        columns=[f"other_{c}" for c in mlb_other.classes_])

# 合并编码结果
df_all = pd.concat([df_all, agri_df, ind_df, other_df], axis=1)

# 行政区划 one-hot
df_all = pd.get_dummies(df_all, columns=['zone', 'county', 'now_admin'], prefix=['zone', 'county', 'now'])

# 数值转换 + 标准化
env_cols = ['altitude', 'rain', 'temp', 'wind', 'dist_road', 'dist_rail', 'dist_canal', 'dist_ocean', 'dist_river']
for col in env_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

# 检查是否有全为NaN或标准差为0的列
print("以下环境变量列存在 NaN 值：")
print(df_all[env_cols].isna().sum())

print("\n以下环境变量列的标准差为 0（无法标准化）：")
print(df_all[env_cols].std())

# 标准化
scaler = StandardScaler()
df_all[env_cols] = scaler.fit_transform(df_all[env_cols])

# 保存最终向量表
df_all.to_csv(r"E:\【论文】\heritage science\0609Chinese_knowledge_graph\commune_vector.csv", index=False, encoding='utf-8-sig')
print("✅ 向量化特征表已保存为 commune_vector.csv")


In [20]:
# 1. 读取你的主向量表（之前生成的）
df_all = pd.read_csv(r"E:\【论文】\heritage science\0609Chinese_knowledge_graph\commune_vector.csv", encoding="utf-8-sig")

# 2. 读取原始环境变量数据
df_env = pd.read_csv(r"E:\【论文】\heritage science\people's commune_nature_feature.csv", encoding="utf-8-sig")  # 请将此路径换成你的真实路径

In [21]:
# 3. 重命名列（确保拼音无冲突）
df_env.rename(columns={
    'Commune': 'name',
    '海拔': 'altitude',
    '降雨': 'rain',
    '温度': 'temp',
    '风力': 'wind',
    '距运河': 'dist_canal',
    '距铁路': 'dist_rail',
    '距海洋': 'dist_ocean',
    '距河流': 'dist_river',
    '距公路': 'dist_road'
}, inplace=True)


In [22]:
# 4. 合并到主表
df_all = pd.merge(df_all, df_env, on='name', how='left')


In [24]:
print(df_all.columns.tolist())

['name', 'location', 'agri_attrs', 'ind_attrs', 'other_attrs', 'altitude_x', 'rain_x', 'temp_x', 'wind_x', 'dist_road_x', 'dist_rail_x', 'dist_canal_x', 'dist_ocean_x', 'dist_river_x', 'lat', 'lon', 'agri_中稻', 'agri_二季', 'agri_人参', 'agri_低产', 'agri_作', 'agri_作物', 'agri_养兔', 'agri_养牛', 'agri_养猪', 'agri_养猪场', 'agri_养畜', 'agri_养禽', 'agri_养蚕', 'agri_养蜂', 'agri_养鱼', 'agri_农具', 'agri_农副业', 'agri_农机具', 'agri_农耕', 'agri_农船', 'agri_制粉', 'agri_副业', 'agri_副业生产', 'agri_化学肥料', 'agri_化肥', 'agri_南瓜', 'agri_名茶', 'agri_土', 'agri_土化', 'agri_土化肥', 'agri_土地', 'agri_土壤', 'agri_圩塘', 'agri_增产', 'agri_士农', 'agri_大蒜', 'agri_大豆', 'agri_大麦', 'agri_大麻', 'agri_太湖银鱼', 'agri_家畜', 'agri_小米', 'agri_小麦', 'agri_山区', 'agri_干菜', 'agri_户麦亩', 'agri_手工业', 'agri_手艺', 'agri_拖拉机', 'agri_捕鱼', 'agri_排灌', 'agri_收获量', 'agri_施肥', 'agri_施肥量', 'agri_早熟', 'agri_春蚕', 'agri_木材', 'agri_木槿', 'agri_木炭', 'agri_木碳', 'agri_杀虫剂', 'agri_杂粮', 'agri_杂谷', 'agri_杉', 'agri_杨树', 'agri_杨槐', 'agri_林业', 'agri_果树', 'agri_果物', 'agri_柳', 'agri_柳梨桃', 'agri_桑

In [25]:
# 1. 删除旧字段（_x）
df_all.drop(columns=[col for col in df_all.columns if col.endswith('_x')], inplace=True)

In [26]:
# 2. 重命名新环境变量字段（去掉 _y 后缀）
df_all.rename(columns={
    'altitude_y': 'altitude',
    'rain_y': 'rain',
    'temp_y': 'temp',
    'wind_y': 'wind',
    'dist_road_y': 'dist_road',
    'dist_rail_y': 'dist_rail',
    'dist_canal_y': 'dist_canal',
    'dist_ocean_y': 'dist_ocean',
    'dist_river_y': 'dist_river'
}, inplace=True)

# 3. 转换为数值并标准化
env_cols = ['altitude', 'rain', 'temp', 'wind', 'dist_road', 'dist_rail', 'dist_canal', 'dist_ocean', 'dist_river']
df_all[env_cols] = df_all[env_cols].apply(pd.to_numeric, errors='coerce')

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_all[env_cols] = scaler.fit_transform(df_all[env_cols])


In [27]:
# 4. 保存最终向量化结果
df_all.to_csv(r"E:\【论文】\heritage science\0609Chinese_knowledge_graph\commune_vector_final.csv", index=False, encoding="utf-8-sig")
print("✅ 成功！环境变量已清洗、标准化并保存为 commune_vector_final.csv")

✅ 成功！环境变量已清洗、标准化并保存为 commune_vector_final.csv
